In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [ ]:
# Initialize system configurations

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Verify BEFORE creating the session

print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()



In [3]:
# Load data from dim_business
df_read_dim_buiness=spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_business")

In [4]:
# Load data from dim_subcategory
df_read_dim_subcategory = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_subcategory")

In [5]:
# Read raw business data
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [6]:
# Extract the ' business_id ' column from business data
df_read_id_categories= df_read_business_data.select(F.col("business_id"),F.col("categories"))

In [7]:
# This function is used to get subcategory names from the 'categories' column

@F.udf(returnType=ArrayType(StringType())) # Define DataFrame schema
def get_subcategory_names(text):

    if text is None:
       return ["Not Specified"]
    
    list_of_element = text.split(",")
     
    del list_of_element[0]

    if len(list_of_element) == 0:
        return  ["Not Specified"]
    list_of_element_without_space = [st.lower().strip() for st in list_of_element]

    return list_of_element_without_space



# Step 1: apply UDF, extract fields, then CACHE + materialize 

df_udf = (
    df_read_id_categories
    .withColumn("subcategory_name_list", get_subcategory_names(F.col("categories")))
    .drop("new_eilte", "categories")
    .cache()
)

df_udf.count()  # Materialization happens at this stage


# Step 2: explode AFTER the UDF stage is fully materialized

df_subcategory_name = (
    df_udf
    .withColumn("subcategory_name", F.explode(F.col("subcategory_name_list")))
    .drop("subcategory_name_list")
)

In [8]:
# Select 'business_id' and 'subcategory_name' columns
df_business_source =  df_subcategory_name.select(F.col("business_id").alias("business_id_row_source") ,F.col("subcategory_name").alias("subcategory_names") )

In [9]:
# Join raw business data with dim_buiness  to get business_id surrogate key

df_join_to_get_business_id = df_business_source.join(df_read_dim_buiness , on = df_read_dim_buiness.Business_id == df_business_source.business_id_row_source , how="inner") 

In [14]:
# Select 'business_id' and 'subcategory_name' and Business_pk_sk columns

df_business = df_join_to_get_business_id.select(F.col("Business_id"),F.col("subcategory_names"),F.col("Business_pk_sk"))

In [17]:
# Join  business data with dim_subcategory  to get subcategory surrogate key

df_join_business_subcategory = df_business.join(df_read_dim_subcategory , on = df_business.subcategory_names == df_read_dim_subcategory.Subcategory_name ,how= "inner" )

In [ ]:
# Select 'Business_pk_sk' and 'Subcategory_pk_sk' columns

df_business_subcategory_bridge = df_join_business_subcategory.select(F.col("Business_pk_sk"),F.col("Subcategory_pk_sk"))

In [10]:
#df_join.select(F.col("business_id"),F.col("subcategory_name"),F.col("Business_pk_sk")).filter(F.col("business_id").isin("Pns2l4eNsfO8kk83dixA6A")).show()

In [ ]:
#df_join_to_get_business_id.show()

In [15]:
#df_business.show()

In [ ]:
#df_join_business_subcategory.filter(F.col("business_id").isin("Pns2l4eNsfO8kk83dixA6A")).show()

In [20]:
#df_business_subcategory_bridge.orderBy(F.col("Business_pk_sk")).show()

In [21]:
#df_business_subcategory_bridge.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/business_subcategory_bridge")